In [1]:
# !pip install -q transformers torch accelerate gradio bitsandbytes

In [2]:
# import torch
# import gradio as gr
# from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
# from google.colab import userdata
# import traceback

# # 1. Fetch your securely stored Hugging Face token
# try:
#     hf_token = userdata.get('HF_TOKEN')
# except userdata.SecretNotFoundError:
#     raise ValueError("You did not save the HF_TOKEN in the Colab Secrets manager correctly.")

# # 2. Configure 4-bit Quantization to fit in the T4 GPU
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16
# )

# # 3. Load the Model and Processor
# model_id = "google/medgemma-1.5-4b-it"

# print("Downloading MedGemma 1.5 (4B) weights... This will take several minutes.")
# processor = AutoProcessor.from_pretrained(model_id, token=hf_token)
# model = AutoModelForImageTextToText.from_pretrained(
#     model_id,
#     quantization_config=bnb_config,
#     device_map="auto",
#     token=hf_token
# )
# print("Model loaded successfully.")

# import traceback

# # 4. Define the Inference Logic (Gemma 3 Architecture Format)
# def analyze_medical_scan(image, text_prompt):
#     try:
#         if image is None:
#             return "Error: You must upload an image."
#         if not text_prompt:
#             text_prompt = "Describe the findings in this medical image."

#         # Ensure 3-channel RGB for the SigLIP encoder
#         image = image.convert("RGB")

#         # GEMMA 3 CHAT TEMPLATE FORMAT
#         # We must pass the image explicitly inside the content dictionary
#         messages = [
#             {
#                 "role": "user",
#                 "content": [
#                     {"type": "image", "image": image},
#                     {"type": "text", "text": text_prompt}
#                 ]
#             }
#         ]

#         # The apply_chat_template function correctly maps the image into the tokenizer's sequence
#         inputs = processor.apply_chat_template(
#             messages,
#             add_generation_prompt=True,
#             tokenize=True,
#             return_dict=True,
#             return_tensors="pt"
#         )

#         # Move inputs to GPU. The specialized processor .to() method handles mixed dtypes correctly here.
#         inputs = inputs.to(model.device, dtype=torch.float16)

#         # Generate the response
#         with torch.no_grad():
#             outputs = model.generate(**inputs, max_new_tokens=512)

#         # We must slice the output to remove the prompt tokens, otherwise it repeats the prompt back to us
#         input_len = inputs["input_ids"].shape[-1]
#         generated_tokens = outputs[0][input_len:]

#         response = processor.decode(generated_tokens, skip_special_tokens=True)
#         return response

#     except Exception as e:
#         error_trace = traceback.format_exc()
#         return f"BACKEND CRASH DETECTED:\n\n{error_trace}"

# # 5. Build the UI
# interface = gr.Interface(
#     fn=analyze_medical_scan,
#     inputs=[
#         gr.Image(type="pil", label="Upload Medical Scan"),
#         gr.Textbox(lines=2, placeholder="e.g., What pathology is visible?")
#     ],
#     # FIX: Expanded the output box to 10 lines to prevent forced scrolling
#     outputs=gr.Textbox(label="MediGemma-X Output", lines=10),
#     title="MediGemma-X MVP (Stage 1)",
#     description="Basic wrapper testing MedGemma 1.5."
# )

# interface.launch(share=True)

In [ ]:
!pip install -q transformers torch accelerate gradio bitsandbytes pillow

import torch
import gradio as gr
import re
import traceback
from PIL import ImageDraw
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from google.colab import userdata

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.0 MB/s eta 0:00:00


In [ ]:
try:
    hf_token = userdata.get('HF_TOKEN')
except userdata.SecretNotFoundError:
    raise ValueError("HF_TOKEN missing in Colab Secrets.")

# 2. Configure 4-bit Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# 3. Load Model
model_id = "google/medgemma-1.5-4b-it"
print("Loading model...")
processor = AutoProcessor.from_pretrained(model_id, token=hf_token)
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token
)
print("Model loaded.")

Loading model...


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


...

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
import gradio as gr
import nest_asyncio
from fastapi.middleware.cors import CORSMiddleware
from PIL import Image, ImageDraw
import traceback
import base64
from io import BytesIO
import torch

nest_asyncio.apply()

def analyze_medical_scan(image_b64, text_prompt):
    print("🚀 ANALYSIS REQUEST RECEIVED")
    try:
        if not image_b64 or image_b64 == '': 
            print("❌ No image data received")
            return None, "No image uploaded."
        
        print(f"✅ Base64 String Received (Length: {len(image_b64)})")
        
        # Decode Base64 to PIL Image
        if "," in image_b64:
            image_b64 = image_b64.split(",")[1]
        image_data = base64.b64decode(image_b64)
        image_rgb = Image.open(BytesIO(image_data)).convert("RGB")
        print(f"📸 Decoded Image Size: {image_rgb.size}")
        
        # 1. MedGemma Diagnosis
        print("🧠 Running MedGemma diagnosis...")
        messages = [{"role": "user", "content": [{"type": "image", "image": image_rgb}, {"type": "text", "text": text_prompt}]}]
        inputs = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors='pt').to(model.device, dtype=torch.float16)
        
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=512)
        
        report = processor.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
        print("✅ Report generated")

        # 2. OWL-ViT Extraction
        print("🎯 Running OWL-ViT grounding...")
        target = "abnormality" 
        for keyword in ["fracture", "pneumonia", "tumor", "lesion", "nodule", "bite", "dermatitis"]: 
            if keyword in report.lower(): target = keyword; break
            
        predictions = detector(image_rgb, candidate_labels=[target], threshold=0.001)
        
        annotated = image_rgb.copy()
        draw = ImageDraw.Draw(annotated)
        
        # Pick the SINGLE best prediction regardless of score
        if len(predictions) > 0:
            best_pred = max(predictions, key=lambda x: x['score'])
            box = best_pred['box']
            draw.rectangle([box['xmin'], box['ymin'], box['xmax'], box['ymax']], outline='red', width=5)
            draw.text((box['xmin'], box['ymin']-15), f"{best_pred['label']} ({round(best_pred['score'],3)})", fill='red')
        
        print("✨ Analysis COMPLETE")
        return annotated, report
    except Exception as e:
        err_msg = traceback.format_exc()
        print(f"❌ ERROR: {err_msg}")
        return None, f"Backend Error: {str(e)}"

with gr.Blocks() as demo:
    gr.Markdown("# MediGemma-X Base64 Backend")
    with gr.Row():
        # CRITICAL: Receive Base64 as a Textbox so Gradio doesn't try to parse it as a file
        img_in = gr.Textbox(label="Base64 Input (Hide in UI)", visible=False)
        txt_in = gr.Textbox()
    with gr.Row():
        img_out = gr.Image(type='pil')
        txt_out = gr.Textbox()
    btn = gr.Button("Run Analysis")
    btn.click(analyze_medical_scan, [img_in, txt_in], [img_out, txt_out], api_name='predict')

# Apply "Nuclear" Security clearance
demo.app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

print("🔗 Starting Base64 Backend with full CORS...")
demo.launch(share=True, max_threads=1, debug=False, show_error=True)


Loading OWL-ViT for Bounding Box Generation...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/613M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/412 [00:00<?, ?it/s]

OwlViTForObjectDetection LOAD REPORT from: google/owlvit-base-patch32
Key                                         | Status     |  | 
--------------------------------------------+------------+--+-
owlvit.vision_model.embeddings.position_ids | UNEXPECTED |  | 
owlvit.text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

preprocessor_config.json:   0%|          | 0.00/392 [00:00<?, ?B/s]

The image processor of type `OwlViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


OWL-ViT loaded.
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3b850fff3211e83fde.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 416, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error


--- VISUAL TARGET EXTRACTED: 'red rash' ---


In [ ]:
# # --- RUN THIS CELL ONLY AFTER THE MODEL CELL IS LOADED ---
# from transformers import pipeline
# import torch
# import gradio as gr
# import traceback
# import re
# from PIL import ImageDraw

# print("Loading OWL-ViT for Bounding Box Generation...")
# detector = pipeline(model="google/owlvit-base-patch32", task="zero-shot-object-detection", device=0)
# print("OWL-ViT loaded.")

# def draw_owlvit_boxes(image, predictions):
#     img_copy = image.copy()
#     draw = ImageDraw.Draw(img_copy)

#     for pred in predictions:
#         box = pred["box"]
#         label = pred["label"]
#         score = pred["score"]

#         xmin, ymin, xmax, ymax = box["xmin"], box["ymin"], box["xmax"], box["ymax"]
#         draw.rectangle([xmin, ymin, xmax, ymax], outline="red", width=4)
#         draw.text((xmin, ymin - 15), f"Auto-Detected: {label}", fill="red")

#     return img_copy

# # --- MAIN INFERENCE LOGIC ---
# def analyze_medical_scan(image, text_prompt):
#     try:
#         if image is None:
#             return None, "Error: You must upload an image."

#         base_prompt = text_prompt if text_prompt else "Analyze this medical scan in detail. Identify the visible anatomy and any potential pathologies."

#         # THE SECRET CHAINING INSTRUCTION
#         # THE STRICT CHAINING INSTRUCTION
#         system_instruction = "\n\nCRITICAL INSTRUCTION: At the very end of your response, you MUST provide a 1-2 word phrase representing ONLY the anomaly, pathology, or injury (e.g., 'rash', 'fracture', 'lesion', 'tumor'). You MUST NOT include the name of the body part or organ. Never say 'foot infection' or 'broken arm', just say 'infection' or 'fracture'. Format it EXACTLY like this: [HIGHLIGHT: <phrase>]. If there is no disease, output [HIGHLIGHT: none]."
#         # system_instruction = "\n\nCRITICAL INSTRUCTION: At the very end of your response, you MUST provide a 1-3 word phrase representing the primary pathology, lesion, or injury visible in the image so a secondary model can visually ground it. Format it EXACTLY like this: [HIGHLIGHT: <phrase>]. If there is no disease, output [HIGHLIGHT: none]. Example: [HIGHLIGHT: skin infection] or [HIGHLIGHT: bone fracture]."

#         final_prompt = base_prompt + system_instruction
#         image_rgb = image.convert("RGB")

#         # STAGE 1: MedGemma Diagnosis
#         messages = [
#             {
#                 "role": "user",
#                 "content": [
#                     {"type": "image", "image": image_rgb},
#                     {"type": "text", "text": final_prompt}
#                 ]
#             }
#         ]

#         inputs = processor.apply_chat_template(
#             messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt"
#         ).to(model.device, dtype=torch.float16)

#         with torch.no_grad():
#             outputs = model.generate(**inputs, max_new_tokens=768)

#         input_len = inputs["input_ids"].shape[-1]
#         generated_tokens = outputs[0][input_len:]
#         raw_response = processor.decode(generated_tokens, skip_special_tokens=True)

#         # STAGE 2: Autonomous Keyword Extraction
#         highlight_target = None
#         clean_response = raw_response

#         # Look for the exact pattern [HIGHLIGHT: whatever]
#         match = re.search(r'\[HIGHLIGHT:\s*(.*?)\]', raw_response, re.IGNORECASE)

#         if match:
#             extracted_phrase = match.group(1).strip().lower()
#             # Remove the secret tag from the text the user will see
#             clean_response = raw_response.replace(match.group(0), "").strip()

#             if extracted_phrase != "none":
#                 highlight_target = extracted_phrase
#                 print(f"\n--- AUTONOMOUS EXTRACTION SUCCESS: '{highlight_target}' ---")
#             else:
#                 print("\n--- AUTONOMOUS EXTRACTION: No pathology detected by MedGemma. ---")
#         else:
#             print("\n--- AUTONOMOUS EXTRACTION FAILED: MedGemma did not follow formatting instructions. ---")

#         # STAGE 3: OWL-ViT Bounding Box
#         annotated_image = image_rgb
#         if highlight_target:
#             predictions = detector(
#                 image_rgb,
#                 candidate_labels=[highlight_target],
#                 threshold=0.001
#             )

#             if predictions:
#                 predictions.sort(key=lambda x: x["score"], reverse=True)
#                 top_prediction = [predictions[0]]
#                 annotated_image = draw_owlvit_boxes(image_rgb, top_prediction)
#             else:
#                 print(f"OWL-ViT could not visually locate the target: '{highlight_target}'")

#         return annotated_image, clean_response

#     except Exception as e:
#         error_trace = traceback.format_exc()
#         return image, f"BACKEND CRASH DETECTED:\n\n{error_trace}"

# # --- USER INTERFACE ---
# with gr.Blocks(title="MediGemma-X Advanced") as interface:
#     gr.Markdown("## MediGemma-X Autonomous Diagnostic Agent\nUpload a scan. The system will diagnose the image and autonomously ground the primary pathology.")

#     with gr.Row():
#         with gr.Column():
#             img_input = gr.Image(type="pil", label="Upload Medical Scan")
#             prompt_input = gr.Textbox(lines=2, label="Diagnostic Query", placeholder="e.g., What is the disease present and the general treatments?")
#             # TARGET INPUT REMOVED - FULLY AUTOMATED NOW
#             submit_btn = gr.Button("Analyze & Auto-Ground", variant="primary")

#         with gr.Column():
#             img_output = gr.Image(type="pil", label="Visual Grounding (Autonomous OWL-ViT)")
#             text_output = gr.Textbox(label="AI Diagnostic Report", lines=15)

#     submit_btn.click(
#         fn=analyze_medical_scan,
#         inputs=[img_input, prompt_input],
#         outputs=[img_output, text_output]
#     )

# interface.launch(share=True, debug=True)

# # # --- RUN THIS CELL ONLY AFTER THE MODEL CELL IS LOADED ---
# # from transformers import pipeline
# # import torch
# # import gradio as gr
# # import traceback
# # from PIL import ImageDraw

# # print("Loading OWL-ViT for Bounding Box Generation...")
# # # Load the zero-shot object detector onto the GPU
# # detector = pipeline(model="google/owlvit-base-patch32", task="zero-shot-object-detection", device=0)
# # print("OWL-ViT loaded.")

# # def draw_owlvit_boxes(image, predictions):
# #     img_copy = image.copy()
# #     draw = ImageDraw.Draw(img_copy)

# #     for pred in predictions:
# #         box = pred["box"]
# #         label = pred["label"]
# #         score = pred["score"]

# #         # We already filtered for the top prediction in the main logic, so we just draw it.
# #         xmin, ymin, xmax, ymax = box["xmin"], box["ymin"], box["xmax"], box["ymax"]
# #         draw.rectangle([xmin, ymin, xmax, ymax], outline="red", width=4)

# #         # Draw the label and the raw confidence score so you can see how the model thinks
# #         draw.text((xmin, ymin - 10), f"{label} ({round(score, 4)})", fill="red")

# #     return img_copy

# # # --- MAIN INFERENCE LOGIC ---
# # def analyze_medical_scan(image, text_prompt, highlight_target):
# #     try:
# #         if image is None:
# #             return None, "Error: You must upload an image."

# #         base_prompt = text_prompt if text_prompt else "Analyze this medical scan in detail. Identify the visible anatomy and any potential pathologies."
# #         image_rgb = image.convert("RGB")

# #         # STAGE 1: MedGemma Diagnosis
# #         messages = [
# #             {
# #                 "role": "user",
# #                 "content": [
# #                     {"type": "image", "image": image_rgb},
# #                     {"type": "text", "text": base_prompt}
# #                 ]
# #             }
# #         ]

# #         inputs = processor.apply_chat_template(
# #             messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt"
# #         ).to(model.device, dtype=torch.float16)

# #         with torch.no_grad():
# #             outputs = model.generate(**inputs, max_new_tokens=768)

# #         input_len = inputs["input_ids"].shape[-1]
# #         generated_tokens = outputs[0][input_len:]
# #         response = processor.decode(generated_tokens, skip_special_tokens=True)

# #         # STAGE 2: OWL-ViT Bounding Box
# #         annotated_image = image_rgb
# #         if highlight_target and highlight_target.strip():
# #             target = highlight_target.strip()
# #             print(f"\n--- HUNTING FOR: {target} ---")

# #             # FORCE pipeline to return low confidence scores
# #             predictions = detector(
# #                 image_rgb,
# #                 candidate_labels=[target],
# #                 threshold=0.001
# #             )

# #             print("--- RAW PREDICTIONS ---")
# #             print(predictions)

# #             if predictions:
# #                 # Sort by highest score descending
# #                 predictions.sort(key=lambda x: x["score"], reverse=True)
# #                 # Take ONLY the absolute best guess
# #                 top_prediction = [predictions[0]]
# #                 annotated_image = draw_owlvit_boxes(image_rgb, top_prediction)
# #             else:
# #                 print("OWL-ViT found absolutely nothing, even with lowered threshold.")

# #         return annotated_image, response

# #     except Exception as e:
# #         error_trace = traceback.format_exc()
# #         return image, f"BACKEND CRASH DETECTED:\n\n{error_trace}"

# # # --- USER INTERFACE ---
# # with gr.Blocks(title="MediGemma-X Advanced") as interface:
# #     gr.Markdown("## MediGemma-X Diagnostic Assistant (Dual-Model Architecture)\nUses MedGemma for text analysis and OWL-ViT for zero-shot spatial grounding.")

# #     with gr.Row():
# #         with gr.Column():
# #             img_input = gr.Image(type="pil", label="Upload Medical Scan")
# #             prompt_input = gr.Textbox(lines=2, label="Diagnostic Query", placeholder="e.g., What is the organ shown?")
# #             target_input = gr.Textbox(lines=1, label="Feature to Highlight (Optional)", placeholder="e.g., bone")
# #             submit_btn = gr.Button("Analyze & Ground", variant="primary")

# #         with gr.Column():
# #             img_output = gr.Image(type="pil", label="Visual Grounding (OWL-ViT)")
# #             text_output = gr.Textbox(label="AI Diagnostic Report (MedGemma)", lines=15)

# #     submit_btn.click(
# #         fn=analyze_medical_scan,
# #         inputs=[img_input, prompt_input, target_input],
# #         outputs=[img_output, text_output]
# #     )

# # interface.launch(share=True, debug=True)

Loading OWL-ViT for Bounding Box Generation...


Loading weights:   0%|          | 0/412 [00:00<?, ?it/s]

OwlViTForObjectDetection LOAD REPORT from: google/owlvit-base-patch32
Key                                         | Status     |  | 
--------------------------------------------+------------+--+-
owlvit.vision_model.embeddings.position_ids | UNEXPECTED |  | 
owlvit.text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


OWL-ViT loaded.
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://756140acb53060b1ac.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 416, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

...


--- AUTONOMOUS EXTRACTION FAILED: MedGemma did not follow formatting instructions. ---


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 416, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error


--- AUTONOMOUS EXTRACTION SUCCESS: 'fracture' ---


Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.



--- AUTONOMOUS EXTRACTION SUCCESS: '<phrase>' ---
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://756140acb53060b1ac.gradio.live


In [ ]:
# # --- UTILITY: BOUNDING BOX EXTRACTION ---
# def draw_bounding_box(image, text_output):
#     # Create a copy so we do not mutate the original input
#     img_copy = image.copy()
#     draw = ImageDraw.Draw(img_copy)

#     # Regex to capture [ymin, xmin, ymax, xmax]
#     pattern = r"\[(\d+),\s*(\d+),\s*(\d+),\s*(\d+)\]"
#     matches = re.findall(pattern, text_output)

#     width, height = img_copy.size

#     for match in matches:
#         ymin, xmin, ymax, xmax = map(int, match)

#         # Denormalize coordinates (assuming standard 0-1000 scale)
#         abs_xmin = (xmin / 1000.0) * width
#         abs_ymin = (ymin / 1000.0) * height
#         abs_xmax = (xmax / 1000.0) * width
#         abs_ymax = (ymax / 1000.0) * height

#         # Draw red box, 4px thickness
#         draw.rectangle([abs_xmin, abs_ymin, abs_xmax, abs_ymax], outline="red", width=4)

#     return img_copy

# # 4. Inference Logic
# def analyze_medical_scan(image, text_prompt):
#     try:
#         if image is None:
#             return None, "Error: You must upload an image."

#         base_prompt = text_prompt if text_prompt else "Analyze this medical scan and describe the findings."

#         # Secretly force the model to output coordinates without the user asking
#         system_instruction = "\n\nCRITICAL INSTRUCTION: You must identify the primary pathology or organ. You MUST output the bounding box coordinates for the affected area strictly in the format [ymin, xmin, ymax, xmax] scaled from 0 to 1000."
#         final_prompt = base_prompt + system_instruction

#         image = image.convert("RGB")

#         messages = [
#             {
#                 "role": "user",
#                 "content": [
#                     {"type": "image", "image": image},
#                     {"type": "text", "text": final_prompt}
#                 ]
#             }
#         ]

#         inputs = processor.apply_chat_template(
#             messages,
#             add_generation_prompt=True,
#             tokenize=True,
#             return_dict=True,
#             return_tensors="pt"
#         ).to(model.device, dtype=torch.float16)

#         # 768 tokens is the maximum you should use on a T4 GPU without causing severe timeout delays
#         with torch.no_grad():
#             outputs = model.generate(**inputs, max_new_tokens=768)

#         input_len = inputs["input_ids"].shape[-1]
#         generated_tokens = outputs[0][input_len:]
#         response = processor.decode(generated_tokens, skip_special_tokens=True)

#         # Process the image to include the drawn boxes
#         annotated_image = draw_bounding_box(image, response)

#         return annotated_image, response

#     except Exception as e:
#         error_trace = traceback.format_exc()
#         return image, f"BACKEND CRASH DETECTED:\n\n{error_trace}"

# # 5. Build UI (Refactored to gr.Blocks for professional layout)
# with gr.Blocks(title="MediGemma-X MVP") as interface:
#     gr.Markdown("## MediGemma-X MVP (Stage 1)\nUpload a scan. The model will automatically analyze it and mark pathologies.")

#     with gr.Row():
#         # Left Column: Inputs
#         with gr.Column():
#             img_input = gr.Image(type="pil", label="Upload Medical Scan")
#             prompt_input = gr.Textbox(lines=2, label="Optional Prompt", placeholder="Leave blank for automatic analysis, or ask a specific question.")
#             submit_btn = gr.Button("Analyze", variant="primary")

#         # Right Column: Outputs
#         with gr.Column():
#             img_output = gr.Image(type="pil", label="Annotated Scan (Auto-Generated)")
#             text_output = gr.Textbox(label="MediGemma-X Output", lines=15)

#     # Connect logic
#     submit_btn.click(
#         fn=analyze_medical_scan,
#         inputs=[img_input, prompt_input],
#         outputs=[img_output, text_output]
#     )

# interface.launch(share=True)

In [ ]:
# # --- RUN THIS CELL ONLY AFTER THE MODEL CELL IS LOADED ---
# from transformers import pipeline
# import torch
# import gradio as gr
# import traceback
# from PIL import ImageDraw

# print("Loading OWL-ViT for Bounding Box Generation...")
# # Load the zero-shot object detector onto the GPU (device=0)
# # This model is small enough to fit alongside 4-bit MedGemma
# detector = pipeline(model="google/owlvit-base-patch32", task="zero-shot-object-detection", device=0)
# print("OWL-ViT loaded.")

# def draw_owlvit_boxes(image, predictions):
#     img_copy = image.copy()
#     draw = ImageDraw.Draw(img_copy)

#     # DEBUGGING: Print exactly what the model output
#     print(f"\n--- RAW OWL-ViT PREDICTIONS ---")
#     print(predictions)
#     print("-------------------------------\n")

#     for pred in predictions:
#         box = pred["box"]
#         label = pred["label"]
#         score = pred["score"]

#         # LOWERED THRESHOLD FOR DEBUGGING
#         if score > 0.01:
#             xmin, ymin, xmax, ymax = box["xmin"], box["ymin"], box["xmax"], box["ymax"]
#             draw.rectangle([xmin, ymin, xmax, ymax], outline="red", width=4)
#             # Added more precision to the printed score
#             draw.text((xmin, ymin - 10), f"{label} ({round(score, 3)})", fill="red")

#     return img_copy

# # --- MAIN INFERENCE LOGIC ---
# def analyze_medical_scan(image, text_prompt, highlight_target):
#     try:
#         if image is None:
#             return None, "Error: You must upload an image."

#         base_prompt = text_prompt if text_prompt else "Analyze this medical scan in detail. Identify the visible anatomy and any potential pathologies."
#         image_rgb = image.convert("RGB")

#         # STAGE 1: MedGemma Diagnosis
#         messages = [
#             {
#                 "role": "user",
#                 "content": [
#                     {"type": "image", "image": image_rgb},
#                     {"type": "text", "text": base_prompt}
#                 ]
#             }
#         ]

#         inputs = processor.apply_chat_template(
#             messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt"
#         ).to(model.device, dtype=torch.float16)

#         with torch.no_grad():
#             outputs = model.generate(**inputs, max_new_tokens=768)

#         input_len = inputs["input_ids"].shape[-1]
#         generated_tokens = outputs[0][input_len:]
#         response = processor.decode(generated_tokens, skip_special_tokens=True)

#         # STAGE 2: OWL-ViT Bounding Box
#         annotated_image = image_rgb
#         if highlight_target and highlight_target.strip():
#             # Run the zero-shot detector looking for the specific keyword
#             predictions = detector(
#                 image_rgb,
#                 candidate_labels=[highlight_target.strip()]
#             )
#             annotated_image = draw_owlvit_boxes(image_rgb, predictions)

#         return annotated_image, response

#     except Exception as e:
#         error_trace = traceback.format_exc()
#         return image, f"BACKEND CRASH DETECTED:\n\n{error_trace}"

# # --- USER INTERFACE ---
# with gr.Blocks(title="MediGemma-X Advanced") as interface:
#     gr.Markdown("## MediGemma-X Diagnostic Assistant (Dual-Model Architecture)\nUses MedGemma for text analysis and OWL-ViT for zero-shot spatial grounding.")

#     with gr.Row():
#         with gr.Column():
#             img_input = gr.Image(type="pil", label="Upload Medical Scan")
#             prompt_input = gr.Textbox(lines=2, label="Diagnostic Query", placeholder="e.g., What is the organ shown?")
#             # NEW UI ELEMENT FOR BOUNDING BOXES
#             target_input = gr.Textbox(lines=1, label="Feature to Highlight (Optional)", placeholder="e.g., fracture, tumor, effusion")
#             submit_btn = gr.Button("Analyze & Ground", variant="primary")

#         with gr.Column():
#             img_output = gr.Image(type="pil", label="Visual Grounding (OWL-ViT)")
#             text_output = gr.Textbox(label="AI Diagnostic Report (MedGemma)", lines=15)

#     submit_btn.click(
#         fn=analyze_medical_scan,
#         inputs=[img_input, prompt_input, target_input],
#         outputs=[img_output, text_output]
#     )

# interface.launch(share=True)

Loading OWL-ViT for Bounding Box Generation...


Loading weights:   0%|          | 0/412 [00:00<?, ?it/s]

OwlViTForObjectDetection LOAD REPORT from: google/owlvit-base-patch32
Key                                         | Status     |  | 
--------------------------------------------+------------+--+-
owlvit.text_model.embeddings.position_ids   | UNEXPECTED |  | 
owlvit.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


OWL-ViT loaded.
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://00b5ca15284d6a15a3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
